In [1]:
import os
import random
import sys
import cv2
import tqdm
import json
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import multilabel_confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import StratifiedShuffleSplit, train_test_split
import albumentations as A
from albumentations.pytorch import ToTensorV2

In [2]:
!pip install polars seaborn

import polars as pl

In [3]:
!nvidia-smi

Sat Apr 25 12:04:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.211.01             Driver Version: 570.211.01     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A5000               On  |   00000000:D1:00.0 Off |                  Off |
| 37%   64C    P2            182W /  230W |   14652MiB /  24564MiB |     15%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
import os
os.chdir('/workspace')

In [5]:
df = pl.read_csv("vindr_mammogram/mammo_processed_merged1/mammo_processed_merged.csv")
df = df.to_pandas()
df["merged_image_path"] = (
    df["merged_image_path"]
    .str.replace("vindr_original_data", "vindr_mammogram", regex=False)
    .str.replace("vindr_original_data", "vindr_mammogram", regex=False)
)

In [6]:
df['cc_finding_categories'].value_counts().sort_index()

cc_finding_categories
['Architectural Distortion', 'Mass']                                                                   1
['Architectural Distortion']                                                                          40
['Asymmetry', 'Mass']                                                                                  1
['Asymmetry']                                                                                         20
['Focal Asymmetry']                                                                                  107
['Global Asymmetry']                                                                                  11
['Mass']                                                                                             443
['Nipple Retraction', 'Mass']                                                                          1
['Nipple Retraction', 'Skin Thickening', 'Mass']                                                       1
['Nipple Retraction']            

In [7]:
def get_combined_finding_6class(cc_findings, mlo_findings, cc_birads, mlo_birads):
    if isinstance(cc_findings, str):
        cc_findings = eval(cc_findings) if cc_findings else []
    if isinstance(mlo_findings, str):
        mlo_findings = eval(mlo_findings) if mlo_findings else []
    
    cc_findings = cc_findings or []
    mlo_findings = mlo_findings or []
    all_findings = set(cc_findings + mlo_findings)
    
    if len(all_findings) > 1 and 'No Finding' in all_findings:
        all_findings.remove('No Finding')
    
    high_suspicion_structural = {
        'Architectural Distortion',
        'Skin Thickening',
        'Skin Retraction',
        'Nipple Retraction'
    }
    
    asymmetry_findings = {
        'Focal Asymmetry',
        'Global Asymmetry',
        'Asymmetry'
    }
    
    has_mass = 'Mass' in all_findings
    has_calc = 'Suspicious Calcification' in all_findings
    has_structural = bool(all_findings & high_suspicion_structural)
    has_asymmetry = bool(all_findings & asymmetry_findings)
    has_lymph = 'Suspicious Lymph Node' in all_findings
    
    def parse_birads(birads_str):
        if pd.isna(birads_str) or birads_str == '':
            return 0
        if isinstance(birads_str, str):
            try:
                return int(birads_str.strip().split()[-1])
            except:
                return 0
        return int(birads_str)
    
    cc_birads_num = parse_birads(cc_birads)
    mlo_birads_num = parse_birads(mlo_birads)
    max_birads = max(cc_birads_num, mlo_birads_num)
    
    if not all_findings or all_findings == {'No Finding'}:
        if max_birads == 1:
            return 0
        elif max_birads == 2:
            return 1
        else:
            return 1 if max_birads == 3 else 4
    
    if (has_mass and has_calc) or has_lymph:
        return 4  # Complex/Combined
    
    # Priority 5: Architectural distortions (without mass)
    if has_structural:
        return 5
    
    # Priority 3: Mass (isolated)
    if has_mass:
        return 3
    
    # Priority 2: Calcification (isolated)
    if has_calc:
        return 2
    
    if has_asymmetry and len(all_findings) == 1:
        return -1
    
    if has_asymmetry and len(all_findings) > 1:
        return 5
    
    print(f"Warning: Unknown finding combination: {all_findings}, BIRADS: {max_birads}")
    return 5

df['finding'] = df.apply(
    lambda row: get_combined_finding_6class(
        row['cc_finding_categories'], 
        row['mlo_finding_categories'],
        row['cc_breast_birads'],
        row['mlo_breast_birads']
    ),
    axis=1
)
df.drop(df[df['finding'] == -1].index, inplace=True)
df['finding'].value_counts().sort_index()

finding
0    6703
1    2329
2     127
3     483
4      85
5      83
Name: count, dtype: int64

In [8]:
import ast
import pandas as pd
from collections import Counter

ASYMMETRY_FINDINGS  = frozenset({"Asymmetry", "Focal Asymmetry", "Global Asymmetry"})
STRUCTURAL_FINDINGS = frozenset({"Architectural Distortion", "Skin Thickening", "Skin Retraction", "Nipple Retraction"})
FINDING_TO_BINARY   = [0, 1, 1, 1, 1, 1]
NUM_PROTOTYPES      = 6

def _parse_findings(raw):
    if raw is None or (isinstance(raw, float) and pd.isna(raw)):
        return frozenset()
    if isinstance(raw, (list, tuple, set)):
        return frozenset(str(f).strip() for f in raw if str(f).strip())
    if isinstance(raw, str):
        raw = raw.strip()
        if not raw:
            return frozenset()
        try:
            parsed = ast.literal_eval(raw)
            if isinstance(parsed, (list, tuple, set)):
                return frozenset(str(f).strip() for f in parsed if str(f).strip())
        except (ValueError, SyntaxError):
            pass
        return frozenset({raw})
    return frozenset()

def _parse_birads(raw):
    if raw is None or (isinstance(raw, float) and pd.isna(raw)):
        return 0
    if isinstance(raw, (int, float)):
        return int(raw)
    s = str(raw).strip()
    for token in reversed(s.split()):
        digits = "".join(c for c in token if c.isdigit())
        if digits:
            return int(digits[0])
    return 0

def add_finding_columns(df,
                        cc_findings_col="cc_finding_categories",
                        mlo_findings_col="mlo_finding_categories",
                        cc_birads_col="cc_breast_birads",
                        mlo_birads_col="mlo_breast_birads"):
    rows, drop_idx, other_log = [], [], []

    for idx, row in df.iterrows():
        cc       = _parse_findings(row[cc_findings_col])
        mlo      = _parse_findings(row[mlo_findings_col])
        combined = cc | mlo

        if len(combined) > 1 and "No Finding" in combined:
            combined = combined - {"No Finding"}

        non_asym = combined - ASYMMETRY_FINDINGS
        if combined and not non_asym:
            drop_idx.append(idx)
            continue
        combined = non_asym

        max_birads  = max(_parse_birads(row[cc_birads_col]), _parse_birads(row[mlo_birads_col]))
        is_negative = not combined or combined == {"No Finding"}

        KNOWN = {"No Finding", "Mass", "Suspicious Calcification", "Suspicious Lymph Node"}
        remaining = combined - KNOWN - STRUCTURAL_FINDINGS

        if not is_negative and remaining:
            other_log.extend(list(remaining))

        rows.append({
            "idx":                idx,
            "has_neg_b1":         int(is_negative and max_birads <= 1),
            "has_neg_b2":         int(is_negative and max_birads > 1),
            "has_mass":           int("Mass" in combined),
            "has_calc":           int("Suspicious Calcification" in combined),
            "has_structural":     int(bool(combined & STRUCTURAL_FINDINGS)),
            "has_lymph": int(not is_negative and (
                                      "Suspicious Lymph Node" in combined or bool(remaining))),
        })

    print("=== Top findings landing in has_lymph_or_other ===")
    for finding, count in Counter(other_log).most_common(20):
        print(f"  {finding}: {count}")

    encoded = pd.DataFrame(rows).set_index("idx")
    df = df.drop(index=drop_idx).join(encoded).reset_index(drop=True)
    return df

df = add_finding_columns(df)

FINDING_COLS = ["has_neg_b1", "has_neg_b2", "has_mass", "has_calc", "has_structural", "has_lymph"]
print("\n=== Finding counts ===")
print(df[FINDING_COLS].sum())


=== Top findings landing in has_lymph_or_other ===

=== Finding counts ===
has_neg_b1        6703
has_neg_b2        2329
has_mass           570
has_calc           207
has_structural      90
has_lymph           22
dtype: int64


In [9]:
inbreast_df = pd.read_csv("inbreast_data/INbreast_merged/merged_metadata.csv")
inbreast_df["merged_image_path"] = (
    inbreast_df["merged_image_path"]
    .str.replace("INbreast Release 1.0", "inbreast_data", regex=False)
    .str.replace("vindr_original_data", "inbreast_data", regex=False))
inbreast_df['birads'] = inbreast_df['birads'].replace({'4a': '4', '4b': '4', '4c': '4','6':'5'})
inbreast_df['label'] = (inbreast_df['birads'].astype(int) - 1).astype(int)
inbreast_df['density'] = 0
inbreast_df['finding'] = 0
inbreast_df['cc_breast_birads'] = None
inbreast_df['mlo_breast_birads'] = None
inbreast_df['cc_breast_density'] = None
inbreast_df['device_id'] = 0
for col in ["has_neg_b1", "has_neg_b2", "has_mass", "has_calc", "has_structural", "has_lymph", "has_other"]:
    inbreast_df[col] = 0
inbreast_df.head()

,patient_id,laterality,merged_image_path,cc_file_name,mlo_file_name,cc_image_path,mlo_image_path,birads,cc_roi_width,cc_roi_height,...,mlo_breast_birads,cc_breast_density,device_id,has_neg_b1,has_neg_b2,has_mass,has_calc,has_structural,has_lymph,has_other
0,024ee3569b2605dc,L,inbreast_data/INbreast_merged/024ee3569b2605dc...,20588020,20588072,INbreast Release 1.0/INbreast_processed/205880...,INbreast Release 1.0/INbreast_processed/205880...,2,1557,3231,...,None,None,0,0,0,0,0,0,0,0
1,024ee3569b2605dc,R,inbreast_data/INbreast_merged/024ee3569b2605dc...,20587994,20588046,INbreast Release 1.0/INbreast_processed/205879...,INbreast Release 1.0/INbreast_processed/205880...,5,1535,3128,...,None,None,0,0,0,0,0,0,0,0
2,069212ec65a94339,L,inbreast_data/INbreast_merged/069212ec65a94339...,50994787,50994733,INbreast Release 1.0/INbreast_processed/509947...,INbreast Release 1.0/INbreast_processed/509947...,1,1226,2580,...,None,None,0,0,0,0,0,0,0,0
3,069212ec65a94339,R,inbreast_data/INbreast_merged/069212ec65a94339...,50994706,50994760,INbreast Release 1.0/INbreast_processed/509947...,INbreast Release 1.0/INbreast_processed/509947...,1,1128,2566,...,None,None,0,0,0,0,0,0,0,0
4,0b7396cdccacca82,L,inbreast_data/INbreast_merged/0b7396cdccacca82...,22670832,22670878,INbreast Release 1.0/INbreast_processed/226708...,INbreast Release 1.0/INbreast_processed/226708...,2,1627,2983,...,None,None,0,0,0,0,0,0,0,0


In [10]:
inbreast_df['label'].value_counts()

label
1    98
0    30
4    28
3    20
2    11
Name: count, dtype: int64

In [11]:
def birads_to_label(birads_category):
    """Convert BI-RADS categories to numerical labels 0-4 (for 5 classes)"""
    birads_num = int(birads_category.replace(" ", "")[-1])
    return birads_num - 1
df['label'] = df['cc_breast_birads'].apply(birads_to_label)

In [12]:
df['label'].value_counts()

label
0    6703
1    2337
3     339
2     319
4     112
Name: count, dtype: int64

In [13]:
df['cc_breast_density'].value_counts()

cc_breast_density
DENSITY C    7486
DENSITY D    1335
DENSITY B     942
DENSITY A      47
Name: count, dtype: int64

In [14]:
data = df[df['split'] == 'training']
test_df = df[df['split'] == 'test']

In [15]:
from sklearn.model_selection import train_test_split


study_meta = (
    data
    .groupby('study_id')
    .agg({
        'cc_breast_birads': 'first',   # BI-RADS at study level
        'finding': 'first'             # finding already encoded as 0–4
    })
    .reset_index()
)


# -------------------------------------------------
study_meta['stratify_key'] = (
    study_meta['cc_breast_birads'].astype(str) + '_' +
    study_meta['finding'].astype(str)
)


train_studies, val_studies = train_test_split(
    study_meta['study_id'],
    test_size=0.1,
    stratify=study_meta['stratify_key'],
    random_state=423
)

train_df = data[data['study_id'].isin(train_studies)].copy()
val_df   = data[data['study_id'].isin(val_studies)].copy()


In [16]:
train_df.shape

(7057, 32)

In [17]:
import numpy as np
import cv2
from PIL import Image
import torchvision.transforms as transforms
import random
import torch


def get_transforms(img_size=(512, 512)):
    """Enhanced mammogram preprocessing with medical imaging considerations"""
    
    train_transform = transforms.Compose([
        transforms.Resize(img_size, interpolation=transforms.InterpolationMode.BICUBIC),
        
        transforms.RandomApply([
            transforms.RandomAffine(
                degrees=10,
                translate=(0.1, 0.1),
                scale=(0.9, 1.1),
                shear=6
            )
        ], p=0.6),
        
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomApply([
            transforms.RandomPerspective(distortion_scale=0.1, p=1.0)
        ], p=0.1),
        
        transforms.RandomApply([
            transforms.ElasticTransform(alpha=5.0, sigma=3.0)
        ], p=0.2),
        transforms.RandomApply([
            transforms.ColorJitter(
                brightness=(0.95, 1.05),
                contrast=(0.9, 1.1)
            )
        ], p=0.4),
        transforms.Lambda(lambda x: adjust_gamma(x, gamma=random.uniform(0.8, 1.2)) 
                         if random.random() < 0.4 else x),
        

        
        transforms.RandomApply([
            transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0))
        ], p=0.2),
        
                # NOISE AUGMENTATIONS
        transforms.Lambda(lambda x: add_gaussian_noise(x, mean=0, std=random.uniform(0.001, 0.02)) 
                         if random.random() < 0.4 else x),
        
        transforms.Lambda(lambda x: add_speckle_noise(x, std=random.uniform(0.01, 0.03)) 
                         if random.random() < 0.2 else x),
        transforms.ToTensor(),
        
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])
    
    val_transform = transforms.Compose([
        transforms.Resize(img_size, interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])
    
    return train_transform, val_transform

def add_gaussian_noise(image, mean=0, std=0.02):
    """Gaussian noise - electronic noise in imaging sensors"""
    if isinstance(image, Image.Image):
        img_array = np.array(image).astype(np.float32) / 255.0
        noise = np.random.normal(mean, std, img_array.shape)
        noisy_img = np.clip(img_array + noise, 0, 1)
        return Image.fromarray((noisy_img * 255).astype(np.uint8))
    return image


def adjust_gamma(image, gamma=1.0):
    """
    Gamma correction - handles tissue density variation
    Gamma < 1 = brighter, > 1 = darker
    """
    if isinstance(image, Image.Image):
        img_array = np.array(image).astype(np.float32) / 255.0
        gamma_corrected = np.power(img_array, gamma)
        return Image.fromarray((gamma_corrected * 255).astype(np.uint8))
    return image

In [18]:
import numpy as np
import random
from PIL import Image
import torchvision.transforms as T

# ── Noise helpers (unchanged — domain-correct) ────────────────────────────────
def add_gaussian_noise(image, mean=0, std=0.02):
    """Electronic sensor noise — additive Gaussian"""
    if isinstance(image, Image.Image):
        arr = np.array(image).astype(np.float32) / 255.0
        noisy = np.clip(arr + np.random.normal(mean, std, arr.shape), 0, 1)
        return Image.fromarray((noisy * 255).astype(np.uint8))
    return image

def add_speckle_noise(image, std=0.03):
    """Multiplicative speckle — physically correct for mammography"""
    if isinstance(image, Image.Image):
        arr = np.array(image).astype(np.float32) / 255.0
        noise = np.random.normal(0, std, arr.shape)
        noisy = np.clip(arr + arr * noise, 0, 1)
        return Image.fromarray((noisy * 255).astype(np.uint8))
    return image

def adjust_gamma(image, gamma=1.0):
    """Gamma correction — tissue density variation simulation"""
    if isinstance(image, Image.Image):
        arr = np.array(image).astype(np.float32) / 255.0
        corrected = np.power(arr, gamma)
        return Image.fromarray((corrected * 255).astype(np.uint8))
    return image

def random_90_rotation(image):
    """
    Only 90°/180°/270° steps — avoids interpolation artifact on
    microcalcifications (Shia et al. 2024, Hamidinekoo et al. 2017)
    """
    k = random.choice([0, 1, 2, 3])
    return image.rotate(k * 90)

def get_transforms(img_size=(512, 512)):

    train_transform = transforms.Compose([
        transforms.Resize(img_size, interpolation=transforms.InterpolationMode.BICUBIC),

        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.2),
        transforms.RandomPerspective(distortion_scale=0.1, p=0.2),
        transforms.RandomApply([
            transforms.ElasticTransform(alpha=15.0, sigma=4.0)
        ], p=0.25),

        transforms.RandomApply([
            transforms.RandomAffine(
                degrees=5,
                translate=None,
                scale=(0.9, 1.05),
                shear=0,
            )
        ], p=0.5),

        # Contrast + brightness — tissue density varies across patients
        transforms.RandomApply([
            transforms.ColorJitter(
                brightness=(0.8, 1.2),
                contrast=(0.7, 1.3),
            )
        ], p=0.5),

        # Gamma — handles exposure variation between machines
        transforms.Lambda(
            lambda x: adjust_gamma(x, gamma=random.uniform(0.8, 1.2))
            if random.random() < 0.4 else x
        ),

        # Gaussian noise — simulates sensor noise, keep it very mild
        transforms.Lambda(
            lambda x: add_gaussian_noise(x, mean=0, std=random.uniform(0.001, 0.02))
            if random.random() < 0.3 else x
        ),

        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ])

    val_transform = transforms.Compose([
        transforms.Resize(img_size, interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ])

    return train_transform, val_transform

In [19]:
import os
import torch
import random
import numpy as np
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

Image.MAX_IMAGE_PIXELS = None
torch.manual_seed(2024)

FINDING_COLS = ["has_neg_b1", "has_neg_b2", "has_mass", "has_calc", "has_structural", "has_lymph"]


class CustomDataset(Dataset):
    def __init__(self, df, transformations=None, ref_transform=None):
        self.df            = df.reset_index(drop=True)
        self.transform     = transformations
        self.ref_transform = ref_transform
        self.cls_names     = {0: "birads_0", 1: "birads_1",2: "birads_2",3: "birads_3",4: "birads_4"}
        self.cls_counts    = df['label'].value_counts().to_dict()

    def __len__(self):
        return len(self.df)

    def get_cls_name(self, label):
        return self.cls_names[label]


    def __getitem__(self, idx):
        row         = self.df.iloc[idx]
        qry_label   = row['label']
        qry_density = row['cc_breast_density']
        qry_finding = row['finding']
        # qry_device  = row['device_id']

        qry_im = Image.open(row['merged_image_path']).convert("RGB")

        if self.transform:
            qry_im = self.transform(qry_im)

        finding_vec = torch.tensor(
            [row[col] for col in FINDING_COLS], dtype=torch.float32
        )
        binary_label = 0 if qry_label == 0 else 1
    
        return {
            "qry_im":      qry_im,
            "qry_gt":      torch.tensor(qry_label,   dtype=torch.long),
             "binary":      torch.tensor(binary_label, dtype=torch.long),
            "finding":     qry_finding,
            "finding_vec": finding_vec,
            "has_neg_b1":     torch.tensor(row["has_neg_b1"],     dtype=torch.long),
            "has_neg_b2":     torch.tensor(row["has_neg_b2"],     dtype=torch.long),
            "has_mass":       torch.tensor(row["has_mass"],       dtype=torch.long),
            "has_calc":       torch.tensor(row["has_calc"],       dtype=torch.long),
            "has_structural": torch.tensor(row["has_structural"], dtype=torch.long),
            "has_lymph":      torch.tensor(row["has_lymph"],      dtype=torch.long)
        }


def get_dls(train_df, val_df, test_df, inbreast_df, bs, ns=6):
    train_trans, val_trans = get_transforms(img_size=(512, 512))

    tr_ds       = CustomDataset(train_df,    train_trans, val_trans)
    vl_ds       = CustomDataset(val_df,      val_trans,   val_trans)
    test_ds     = CustomDataset(test_df,     val_trans,   val_trans)
    inbreast_ds = CustomDataset(inbreast_df, val_trans,   val_trans)

    labels                       = train_df['label'].values
    unique_classes, class_counts = np.unique(labels, return_counts=True)
    beta                         = 0.5
    class_weights                = (1.0 / class_counts) ** beta
    class_weights                = class_weights / class_weights.sum() * len(unique_classes)
    sample_weights               = class_weights[labels]

    print("Class counts:",           dict(zip(unique_classes, class_counts)))
    print("Smoothed class weights:", np.round(class_weights, 3))

    sampler = WeightedRandomSampler(
        weights     = torch.from_numpy(sample_weights).float(),
        num_samples = len(sample_weights),
        replacement = True,
    )

    tr_dl = DataLoader(
        tr_ds, batch_size=bs, 
        shuffle=True,
        # sampler=sampler,
        drop_last=True, num_workers=4, pin_memory=True, persistent_workers=True,
    )
    val_dl = DataLoader(
        vl_ds, batch_size=bs, shuffle=False,
        drop_last=False, num_workers=8, pin_memory=True, persistent_workers=True,
    )
    ts_dl       = DataLoader(test_ds,     batch_size=1, shuffle=False, num_workers=ns)
    inbreast_dl = DataLoader(inbreast_ds, batch_size=1, shuffle=False, num_workers=ns)

    return tr_dl, val_dl, ts_dl, inbreast_dl, tr_ds.cls_names, tr_ds.cls_counts

In [20]:
tr_dl, val_dl, ts_dl, inbreast_dl, classes, cls_counts = get_dls(train_df,val_df, test_df, inbreast_df, bs =16)

Class counts: {np.int64(0): np.int64(4824), np.int64(1): np.int64(1674), np.int64(2): np.int64(229), np.int64(3): np.int64(247), np.int64(4): np.int64(83)}
Smoothed class weights: [0.259 0.439 1.187 1.143 1.972]


In [21]:
# finding columns distribution
finding_cols = ['has_neg_b1','has_neg_b2','has_mass','has_calc','has_structural','has_lymph']
print(train_df[finding_cols].sum())

has_neg_b1        4824
has_neg_b2        1666
has_mass           414
has_calc           146
has_structural      72
has_lymph           18
dtype: int64


In [22]:
# THE KEY — joint distribution: for each finding, how many samples per BI-RADS class
for f in finding_cols:
    print(f"\n--- {f} ---")
    print(train_df[train_df[f]==1]['label'].value_counts().sort_index())


--- has_neg_b1 ---
label
0    4824
Name: count, dtype: int64

--- has_neg_b2 ---
label
1    1666
Name: count, dtype: int64

--- has_mass ---
label
2    195
3    156
4     63
Name: count, dtype: int64

--- has_calc ---
label
2    24
3    78
4    44
Name: count, dtype: int64

--- has_structural ---
label
1     7
2    12
3    39
4    14
Name: count, dtype: int64

--- has_lymph ---
label
1    1
3    9
4    8
Name: count, dtype: int64


In [ ]:
import os
import gc
import random
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

from tqdm import tqdm
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    confusion_matrix,
    classification_report,
)

# =============================================================================
# Config
# =============================================================================

FINDING_COLS = [
    "has_neg_b1",       # finding 0
    "has_neg_b2",       # finding 1
    "has_mass",         # finding 2
    "has_calc",         # finding 3
    "has_structural",   # finding 4
    "has_lymph",        # finding 5
]

NUM_FINDINGS       = 6
NUM_CLASSES        = 5   # BI-RADS 1-5 — used for birads proto path only
BINARY_CLASS_NAMES = ["Negative (BI-RADS 1)", "Positive (BI-RADS 2-5)"]
BIRADS_CLASS_NAMES = ["BI-RADS 1", "BI-RADS 2", "BI-RADS 3", "BI-RADS 4", "BI-RADS 5"]

# =============================================================================
# Architecture — Three heads on shared backbone
#
#   binary_head       → [B, 2]   2-class logits (BI-RADS 1 vs BI-RADS 2-5)
#                                 ONLY classification head
#                                 trained with weighted CrossEntropy
#
#   proj_head_finding → [B, 128] finding embedding space
#                                 6 prototypes, one per finding type
#                                 contrastive regularizer only — not used at inference
#
#   proj_head_birads  → [B, 128] birads embedding space
#                                 5 prototypes, one per BI-RADS class
#                                 contrastive regularizer only — not used at inference
#
# No cls_head — binary_head is the only classification output
# The two proj_heads operate in separate embedding spaces → no gradient conflict
# =============================================================================

# Binary class counts from your data:
#   binary=0 (BI-RADS 1):    4824 samples
#   binary=1 (BI-RADS 2-5): ~2232 samples
# weight[c] = total / (num_classes * count_c)
BINARY_CLASS_WEIGHT = [
    7056.0 / (2.0 * 4824.0),   # 0: BI-RADS 1     ≈ 0.731
    7056.0 / (2.0 * 2232.0),   # 1: BI-RADS 2-5   ≈ 1.581
]

# Finding prototype momentums — indexed by finding (0..5)
# Lower momentum = faster update = better for rare findings
FINDING_PROTO_MOMENTUM = [
    0.999,  # 0: neg_b1      (4824) — massive data
    0.997,  # 1: neg_b2      (1666) — large
    0.980,  # 2: mass        ( 414) — medium
    0.960,  # 3: calc        ( 146) — moderate
    0.920,  # 4: structural  (  72) — sparse
    0.850,  # 5: lymph       (  18) — very sparse, update aggressively
]

# BI-RADS prototype momentums — indexed by label (0..4)
BIRADS_PROTO_MOMENTUM = [
    0.999,  # 0: BI-RADS 1   (4824)
    0.997,  # 1: BI-RADS 2   (1666)
    0.970,  # 2: BI-RADS 3   ( ~414) — hard class
    0.960,  # 3: BI-RADS 4   ( ~270) — hard class
    0.920,  # 4: BI-RADS 5   (  ~63) — rarest
]

# Sample weights for finding proto loss
# neg_b1, neg_b2 → easy majority → low weight
# mass, calc, structural, lymph → hard minority → high weight
FINDING_SAMPLE_WEIGHT = [
    0.1,   # 0: neg_b1
    0.2,   # 1: neg_b2
    1.5,   # 2: mass
    1.5,   # 3: calc
    1.5,   # 4: structural
    1.5,   # 5: lymph
]

BIRADS_SAMPLE_WEIGHT = [
    0.1,   # 0: BI-RADS 1
    0.2,   # 1: BI-RADS 2
    1.5,   # 2: BI-RADS 3
    1.5,   # 3: BI-RADS 4
    1.5,   # 4: BI-RADS 5
]

# Minimum EMA updates before a prototype participates in the loss
PROTO_MIN_UPDATES = 15

# =============================================================================
# Hard Negative Mining & Mixup Config
# =============================================================================
HARD_NEG_MINING_ENABLED = True
HARD_NEG_MINING_START_EPOCH = 5  # Start mining after epoch 5 for stability
HARD_NEG_THRESHOLD = 0.45  # Positive score threshold for "hard negatives"
HARD_NEG_SAMPLE_SIZE = 500  # Max hard negatives to collect per validation
HARD_NEG_WEIGHT_MULTIPLIER = 2.0  # How much to weight hard negatives

MIXUP_ENABLED = True
MIXUP_ALPHA = 1.0  # Beta distribution parameter for mixing
MIXUP_MIN_POSITIVES_PER_BATCH = 2  # Only apply if batch has >= 2 positives


def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


# =============================================================================
# Attention Pooling
# =============================================================================

class AttentionPool2d(nn.Module):
    def __init__(self, in_channels, reduction=4):
        super().__init__()
        h = max(in_channels // reduction, 32)
        self.attn = nn.Sequential(
            nn.Conv2d(in_channels, h, kernel_size=1, bias=False),
            nn.BatchNorm2d(h),
            nn.GELU(),
            nn.Conv2d(h, 1, kernel_size=1, bias=True),
        )

    def forward(self, x):
        w = F.softmax(self.attn(x).flatten(2), dim=-1)
        return (x.flatten(2) * w).sum(-1)


# =============================================================================
# Backbone Encoder
#
#   binary_head       → [B, 2]   classification output
#   proj_head_finding → [B, 128] finding embedding (normalized)
#   proj_head_birads  → [B, 128] birads  embedding (normalized)
# =============================================================================

class FindingAwareProtoModel(nn.Module):
    def __init__(
        self,
        backbone_name,
        backbone,
        emb_dim=128,
        dropout=0.4,
    ):
        super().__init__()
        self.backbone_name = backbone_name.lower()
        self.backbone      = backbone

        if "efficientnet" in self.backbone_name:
            self.num_features = backbone.classifier[1].in_features
            backbone.classifier[1] = nn.Identity()
            self.is_cnn = True
        elif "convnext" in self.backbone_name:
            self.num_features = backbone.classifier[2].in_features
            backbone.classifier[2] = nn.Identity()
            self.is_cnn = True
        elif "resnet" in self.backbone_name:
            self.num_features = backbone.fc.in_features
            backbone.fc = nn.Identity()
            self.is_cnn = True
        elif "densenet" in self.backbone_name:
            self.num_features = backbone.classifier.in_features
            backbone.classifier = nn.Identity()
            self.is_cnn = True
        elif "swin" in self.backbone_name:
            self.num_features = backbone.head.in_features
            backbone.head = nn.Identity()
            self.is_cnn = False
        else:
            raise ValueError(f"Unsupported backbone: {backbone_name}")

        self.pool = AttentionPool2d(self.num_features) if self.is_cnn else None

        # ── Binary classification head — ONLY classification output ──────────
        # 2-class: class 0 = BI-RADS 1 (negative), class 1 = BI-RADS 2-5 (positive)
        self.binary_head = nn.Sequential(
            nn.Linear(self.num_features, 256),
            nn.LayerNorm(256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, 2),              # [B, 2]
        )

        # ── Finding projection head — contrastive regularizer only ───────────
        # Produces 128d normalized embeddings for finding proto InfoNCE loss
        # NOT used at inference time
        self.proj_head_finding = nn.Sequential(
            nn.Linear(self.num_features, self.num_features),
            nn.LayerNorm(self.num_features),
            nn.ReLU(inplace=True),
            nn.Linear(self.num_features, emb_dim),
        )

        # ── BI-RADS projection head — contrastive regularizer only ───────────
        # Produces 128d normalized embeddings for birads proto InfoNCE loss
        # NOT used at inference time
        self.proj_head_birads = nn.Sequential(
            nn.Linear(self.num_features, self.num_features),
            nn.LayerNorm(self.num_features),
            nn.ReLU(inplace=True),
            nn.Linear(self.num_features, emb_dim),
        )

        self._init_weights()

    def _init_weights(self):
        for head in [self.binary_head, self.proj_head_finding, self.proj_head_birads]:
            for m in head.modules():
                if isinstance(m, nn.Linear):
                    nn.init.trunc_normal_(m.weight, std=0.02)
                    if m.bias is not None:
                        nn.init.zeros_(m.bias)
                elif isinstance(m, (nn.LayerNorm, nn.BatchNorm1d)):
                    if hasattr(m, "weight") and m.weight is not None:
                        nn.init.ones_(m.weight)
                    if hasattr(m, "bias") and m.bias is not None:
                        nn.init.zeros_(m.bias)

    def _extract(self, x):
        f = self.backbone(x)
        if isinstance(f, tuple):
            f = f[0]
        if self.is_cnn:
            if f.ndim == 4:
                f = self.pool(f) if self.pool is not None else f.flatten(2).mean(-1)
            elif f.ndim == 3:
                f = f.mean(1)
        else:
            if f.ndim == 3:
                f = f.mean(1)
            elif f.ndim == 4:
                f = f.flatten(2).mean(-1)
        return f

    def forward(self, x, return_embeddings=False):
        feat         = self._extract(x)
        binary_logit = self.binary_head(feat)              # [B, 2]

        if return_embeddings:
            emb_f = F.normalize(self.proj_head_finding(feat), dim=1)  # [B, 128]
            emb_b = F.normalize(self.proj_head_birads(feat),  dim=1)  # [B, 128]
            return binary_logit, emb_f, emb_b

        return binary_logit


# =============================================================================
# Dual-Path Prototype Network
# =============================================================================

class DualProtoNet(nn.Module):
    def __init__(
        self,
        backbone_name,
        backbone_fn,
        backbone_weights,
        emb_dim=128,
        dropout=0.4,
        num_classes=NUM_CLASSES,
        num_findings=NUM_FINDINGS,
        temperature=0.07,
    ):
        super().__init__()
        self.num_classes  = num_classes
        self.num_findings = num_findings
        self.temperature  = temperature
        self.emb_dim      = emb_dim

        backbone = backbone_fn(weights=backbone_weights)
        self.encoder = FindingAwareProtoModel(
            backbone_name=backbone_name,
            backbone=backbone,
            emb_dim=emb_dim,
            dropout=dropout,
        )

        # Finding prototypes [6, D]
        self.register_buffer(
            "proto_finding",
            F.normalize(torch.randn(num_findings, emb_dim), dim=-1),
        )
        self.register_buffer(
            "proto_finding_updates",
            torch.zeros(num_findings, dtype=torch.long),
        )

        # BI-RADS prototypes [5, D]
        self.register_buffer(
            "proto_birads",
            F.normalize(torch.randn(num_classes, emb_dim), dim=-1),
        )
        self.register_buffer(
            "proto_birads_updates",
            torch.zeros(num_classes, dtype=torch.long),
        )

        # Adaptive momentums — move to device automatically as buffers
        self.register_buffer(
            "finding_momentum",
            torch.tensor(FINDING_PROTO_MOMENTUM, dtype=torch.float),
        )
        self.register_buffer(
            "birads_momentum",
            torch.tensor(BIRADS_PROTO_MOMENTUM, dtype=torch.float),
        )

    @torch.no_grad()
    def update_finding_prototypes(self, emb_f, finding_vec):
        """
        emb_f:       [B, D] normalized — from proj_head_finding
        finding_vec: [B, 6] multi-hot
        """
        for f in range(self.num_findings):
            mask = finding_vec[:, f] > 0.5
            if mask.sum() == 0:
                continue
            batch_proto = F.normalize(emb_f[mask].mean(dim=0), dim=0)
            m = self.finding_momentum[f].item()
            if self.proto_finding_updates[f] == 0:
                self.proto_finding[f] = batch_proto
            else:
                self.proto_finding[f] = F.normalize(
                    m * self.proto_finding[f] + (1.0 - m) * batch_proto, dim=0
                )
            self.proto_finding_updates[f] += 1

    @torch.no_grad()
    def update_birads_prototypes(self, emb_b, labels):
        """
        emb_b:  [B, D] normalized — from proj_head_birads
        labels: [B]    int (0..4)
        """
        for c in range(self.num_classes):
            mask = labels == c
            if mask.sum() == 0:
                continue
            batch_proto = F.normalize(emb_b[mask].mean(dim=0), dim=0)
            m = self.birads_momentum[c].item()
            if self.proto_birads_updates[c] == 0:
                self.proto_birads[c] = batch_proto
            else:
                self.proto_birads[c] = F.normalize(
                    m * self.proto_birads[c] + (1.0 - m) * batch_proto, dim=0
                )
            self.proto_birads_updates[c] += 1

    def forward_encoder(self, x, return_embeddings=False):
        return self.encoder(x, return_embeddings=return_embeddings)


# =============================================================================
# InfoNCE Proto Loss
#
# Standard InfoNCE — denominator includes ALL ready prototypes (pos + neg)
# → loss always >= 0 by construction
# Prototypes must reach PROTO_MIN_UPDATES before participating
# =============================================================================

class ProtoInfoNCELoss(nn.Module):
    def __init__(self, temperature=0.07, min_updates=PROTO_MIN_UPDATES):
        super().__init__()
        self.tau         = temperature
        self.min_updates = min_updates

    def forward(self, q, prototypes, proto_updates, sample_weights_list, pos_indices):
        """
        q:                   [B, D] normalized query embeddings
        prototypes:          [P, D] normalized prototype embeddings
        proto_updates:       [P]    int — update count per prototype
        sample_weights_list: [B]    float — per-sample loss weight
        pos_indices:         [B]    int — positive proto index (-1 = skip)
        """
        ready   = (proto_updates >= self.min_updates)
        losses  = []
        weights = []

        for i in range(q.shape[0]):
            pos_idx = pos_indices[i]
            if pos_idx < 0 or not ready[pos_idx]:
                continue

            qi            = q[i]
            all_sims      = []
            pos_local_idx = None

            for p in range(prototypes.shape[0]):
                if not ready[p]:
                    continue
                sim = torch.dot(qi, prototypes[p]) / self.tau
                if p == pos_idx:
                    pos_local_idx = len(all_sims)
                all_sims.append(sim)

            if pos_local_idx is None or len(all_sims) < 2:
                continue

            all_sims    = torch.stack(all_sims)
            log_softmax = F.log_softmax(all_sims, dim=0)
            loss_i      = -log_softmax[pos_local_idx]

            losses.append(loss_i)
            weights.append(sample_weights_list[i])

        if len(losses) == 0:
            return torch.tensor(0.0, device=q.device, requires_grad=False)

        losses  = torch.stack(losses)
        weights = torch.tensor(weights, device=q.device, dtype=q.dtype)
        return (weights * losses).sum() / (weights.sum() + 1e-8)


# =============================================================================
# Validation
# =============================================================================

@torch.no_grad()
def validate(model, loader, device, binary_loss_fn, class_names=None):
    class_names = class_names or BINARY_CLASS_NAMES
    model.eval()

    total_loss             = 0.0
    bin_preds, bin_targets = [], []

    for batch in loader:
        imgs      = batch["qry_im"].to(device, non_blocking=True)
        binary_gt = batch["binary"].to(device, non_blocking=True).long()

        binary_logit = model.forward_encoder(imgs, return_embeddings=False)
        total_loss  += binary_loss_fn(binary_logit, binary_gt).item()

        pred = binary_logit.argmax(1)
        bin_preds.extend(pred.cpu().numpy())
        bin_targets.extend(binary_gt.cpu().numpy())

    acc      = accuracy_score(bin_targets, bin_preds)
    macro_f1 = f1_score(bin_targets, bin_preds, average="macro",    zero_division=0)
    wt_f1    = f1_score(bin_targets, bin_preds, average="weighted", zero_division=0)

    print("\n--- Validation (Binary) ---")
    print(classification_report(bin_targets, bin_preds, target_names=class_names, zero_division=0))

    return {
        "loss":        total_loss / max(len(loader), 1),
        "acc":         acc,
        "macro_f1":    macro_f1,
        "weighted_f1": wt_f1,
    }


# =============================================================================
# Hard Negative Mining
# =============================================================================

@torch.no_grad()
def mine_hard_negatives(model, loader, device, threshold=HARD_NEG_THRESHOLD, max_samples=HARD_NEG_SAMPLE_SIZE):
    """
    Identify hard negatives: samples with positive label that score < threshold
    Returns list of (batch_idx, sample_idx_in_batch, logit_score) tuples
    """
    model.eval()
    hard_negatives = []

    for batch_idx, batch in enumerate(loader):
        imgs      = batch["qry_im"].to(device, non_blocking=True)
        binary_gt = batch["binary"].to(device, non_blocking=True).long()

        binary_logit = model.forward_encoder(imgs, return_embeddings=False)
        pos_probs = binary_logit.softmax(1)[:, 1]  # Probability of positive class

        # Find positives that scored low (hard negatives)
        is_positive = binary_gt == 1
        low_score = pos_probs < threshold
        is_hard = is_positive & low_score

        hard_idx = torch.nonzero(is_hard, as_tuple=False).squeeze(-1)
        for sample_idx in hard_idx:
            hard_negatives.append({
                "batch_idx": batch_idx,
                "sample_idx": sample_idx.item(),
                "score": pos_probs[sample_idx].item(),
            })

    # Sort by score (ascending) and keep top-k hardest
    hard_negatives.sort(key=lambda x: x["score"])
    hard_negatives = hard_negatives[:max_samples]

    print(f"  [Hard Neg Mining] Found {len(hard_negatives)} hard negatives (threshold={threshold:.3f})")
    return hard_negatives


# =============================================================================
# Mixup Augmentation
# =============================================================================

def apply_mixup_to_batch(imgs, binary_gt, finding_vec, alpha=MIXUP_ALPHA):
    """
    Apply mixup to positive samples in the batch.
    Only mixes positives with other positives to preserve labels.
    
    Args:
        imgs: [B, C, H, W]
        binary_gt: [B] long
        finding_vec: [B, 6] float
        alpha: Beta distribution parameter
    
    Returns:
        imgs_mixed, binary_gt (labels stay same for positive-positive mixes)
    """
    pos_mask = binary_gt == 1
    pos_indices = torch.nonzero(pos_mask, as_tuple=False).squeeze(-1)

    if len(pos_indices) < MIXUP_MIN_POSITIVES_PER_BATCH:
        return imgs, binary_gt, finding_vec  # Not enough positives to mix

    # Randomly pair positives
    num_pos = len(pos_indices)
    lam_samples = np.random.beta(alpha, alpha, size=num_pos // 2)

    imgs_mixed = imgs.clone()
    finding_vec_mixed = finding_vec.clone()

    for mix_idx in range(num_pos // 2):
        idx_a = pos_indices[mix_idx].item()
        idx_b = pos_indices[num_pos - 1 - mix_idx].item()
        lam = lam_samples[mix_idx]

        # Mix images
        imgs_mixed[idx_a] = lam * imgs[idx_a] + (1.0 - lam) * imgs[idx_b]
        
        # Mix finding vectors (take union of findings since both are positive)
        finding_vec_mixed[idx_a] = torch.clamp(
            finding_vec[idx_a] + finding_vec[idx_b], 0, 1
        )

    return imgs_mixed, binary_gt, finding_vec_mixed


# =============================================================================
# Test
# =============================================================================

@torch.no_grad()
def test_model(model, loader, device, save_dir, name="Test", class_names=None):
    class_names = class_names or BINARY_CLASS_NAMES
    model.eval()

    bin_preds, bin_targets = [], []

    for batch in loader:
        imgs      = batch["qry_im"].to(device, non_blocking=True)
        binary_gt = batch["binary"].to(device, non_blocking=True).long()

        binary_logit = model.forward_encoder(imgs, return_embeddings=False)
        pred         = binary_logit.argmax(1)

        bin_preds.extend(pred.cpu().numpy())
        bin_targets.extend(binary_gt.cpu().numpy())

    acc      = accuracy_score(bin_targets, bin_preds)
    macro_f1 = f1_score(bin_targets, bin_preds, average="macro",    zero_division=0)
    wt_f1    = f1_score(bin_targets, bin_preds, average="weighted", zero_division=0)
    cm       = confusion_matrix(bin_targets, bin_preds)

    print(f"\n{'='*70}")
    print(f"{name} | Acc={acc:.4f}  Macro-F1={macro_f1:.4f}  Weighted-F1={wt_f1:.4f}")
    print(cm)
    print(classification_report(bin_targets, bin_preds, target_names=class_names, zero_division=0))
    print('=' * 70)

    os.makedirs(save_dir, exist_ok=True)
    with open(os.path.join(save_dir, f"{name.lower().replace(' ', '_')}_report.txt"), "w") as fh:
        fh.write(f"{name}\n")
        fh.write(f"Acc={acc:.4f}  Macro-F1={macro_f1:.4f}  Weighted-F1={wt_f1:.4f}\n\n")
        fh.write(str(cm) + "\n\n")
        fh.write(classification_report(bin_targets, bin_preds, target_names=class_names, zero_division=0))

    return {"acc": acc, "macro_f1": macro_f1, "weighted_f1": wt_f1}


class AsymmetricFocalLoss(nn.Module):
    def __init__(self, gamma_pos=2.0, gamma_neg=2.0, alpha=0.9):
        super().__init__()
        self.gp = gamma_pos; self.gn = gamma_neg; self.alpha = alpha

    def forward(self, logits, targets):
        targets = targets.long()
        ce      = F.cross_entropy(logits, targets, reduction="none")
        pt      = torch.exp(-ce)
        gamma   = torch.where(targets == 1, torch.full_like(ce, self.gp),
                              torch.full_like(ce, self.gn))
        alpha_t = torch.where(targets == 1, torch.full_like(ce, self.alpha),
                              torch.full_like(ce, 1 - self.alpha))
        return (alpha_t * (1 - pt) ** gamma * ce).mean()


def train_model(
    model,
    train_loader,
    val_loader,
    device,
    lr_backbone=5e-5,
    lr_heads=5e-5,
    epochs=60,
    patience=15,
    save_path="best_model.pth",
    w_finding=0.3,
    w_birads=0.3,
    warmup_epochs=3,
    ramp_epochs=10,
    class_names=None,
):
    """
    Enhanced training with:
    - Hard negative mining: identify positives with low confidence, weight them higher
    - Mixup: mix positive samples to create synthetic hard boundary cases
    """
    class_names = class_names or BINARY_CLASS_NAMES
    os.makedirs(os.path.dirname(os.path.abspath(save_path)), exist_ok=True)
    log_path = save_path.replace(".pth", "_log.txt")

    binary_loss_fn = AsymmetricFocalLoss().to(device)

    proto_loss_fn = ProtoInfoNCELoss(
        temperature=model.temperature,
        min_updates=PROTO_MIN_UPDATES,
    ).to(device)

    optimizer = AdamW([
        {
            "params":       model.encoder.backbone.parameters(),
            "lr":           lr_backbone,
            "weight_decay": 0.05,
        },
        {
            "params":       model.encoder.binary_head.parameters(),
            "lr":           lr_heads,
            "weight_decay": 0.05,
        },
        {
            "params":       model.encoder.proj_head_finding.parameters(),
            "lr":           lr_heads,
            "weight_decay": 0.05,
        },
        {
            "params":       model.encoder.proj_head_birads.parameters(),
            "lr":           lr_heads,
            "weight_decay": 0.05,
        },
    ])

    scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    scaler    = torch.amp.GradScaler("cuda") if device.type == "cuda" else None

    best_macro_f1 = 0.0
    not_improved  = 0
    hard_negatives_pool = []  # Pool of hard negatives to weight during training

    for epoch in range(epochs):
        if epoch < warmup_epochs:
            pw_f = 0.0
            pw_b = 0.0
        else:
            ramp = min(1.0, (epoch - warmup_epochs + 1) / max(ramp_epochs, 1))
            pw_f = w_finding * ramp
            pw_b = w_birads  * ramp

        # ── Hard Negative Mining ─────────────────────────────────────────────
        if HARD_NEG_MINING_ENABLED and epoch >= HARD_NEG_MINING_START_EPOCH and epoch % 3 == 0:
            print(f"\n[Epoch {epoch+1}] Running hard negative mining...")
            hard_negatives_pool = mine_hard_negatives(
                model, val_loader, device,
                threshold=HARD_NEG_THRESHOLD,
                max_samples=HARD_NEG_SAMPLE_SIZE
            )

        model.train()
        e_binary = e_proto_f = e_proto_b = 0.0
        all_preds   = []
        all_targets = []

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")

        for batch in pbar:
            imgs        = batch["qry_im"].to(device, non_blocking=True)
            labels      = batch["qry_gt"].to(device, non_blocking=True).long()
            binary_gt   = batch["binary"].to(device, non_blocking=True).long()
            finding_vec = batch["finding_vec"].to(device, non_blocking=True).float()

            # ── Apply Mixup ──────────────────────────────────────────────────
            if MIXUP_ENABLED and random.random() < 0.5:  # 50% chance to apply mixup
                imgs, binary_gt, finding_vec = apply_mixup_to_batch(
                    imgs, binary_gt, finding_vec, alpha=MIXUP_ALPHA
                )

            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast(device_type=device.type, enabled=(scaler is not None)):

                binary_logit, emb_f, emb_b = model.forward_encoder(
                    imgs, return_embeddings=True
                )

                # ── Binary classification loss ───────────────────────────────
                binary_loss = binary_loss_fn(binary_logit, binary_gt)

                # ── Finding proto loss ───────────────────────────────────────
                if pw_f > 0:
                    finding_pos_indices = []
                    finding_sw          = []
                    for i in range(imgs.shape[0]):
                        active = torch.where(finding_vec[i] > 0.5)[0]
                        if len(active) == 0:
                            finding_pos_indices.append(-1)
                            finding_sw.append(0.0)
                        else:
                            fi = active[0].item()
                            finding_pos_indices.append(fi)
                            finding_sw.append(FINDING_SAMPLE_WEIGHT[fi])

                    proto_loss_f = proto_loss_fn(
                        q=emb_f,
                        prototypes=model.proto_finding,
                        proto_updates=model.proto_finding_updates,
                        sample_weights_list=finding_sw,
                        pos_indices=finding_pos_indices,
                    )
                else:
                    proto_loss_f = torch.tensor(0.0, device=device)

                # ── BI-RADS proto loss ───────────────────────────────────────
                if pw_b > 0:
                    birads_pos_indices = labels.cpu().tolist()
                    birads_sw = [
                        BIRADS_SAMPLE_WEIGHT[lb] for lb in birads_pos_indices
                    ]
                    proto_loss_b = proto_loss_fn(
                        q=emb_b,
                        prototypes=model.proto_birads,
                        proto_updates=model.proto_birads_updates,
                        sample_weights_list=birads_sw,
                        pos_indices=birads_pos_indices,
                    )
                else:
                    proto_loss_b = torch.tensor(0.0, device=device)

                total_loss = binary_loss + pw_f * proto_loss_f + pw_b * proto_loss_b

            if scaler is not None:
                scaler.scale(total_loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.encoder.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                total_loss.backward()
                torch.nn.utils.clip_grad_norm_(model.encoder.parameters(), 1.0)
                optimizer.step()

            # ── Prototype EMA update — no grad, fresh embeddings ─────────────
            with torch.no_grad():
                feat = model.encoder._extract(imgs)

                emb_f_fresh = F.normalize(
                    model.encoder.proj_head_finding(feat), dim=1
                )
                model.update_finding_prototypes(emb_f_fresh, finding_vec)

                emb_b_fresh = F.normalize(
                    model.encoder.proj_head_birads(feat), dim=1
                )
                model.update_birads_prototypes(emb_b_fresh, labels)

            pred = binary_logit.argmax(1)
            all_preds.extend(pred.detach().cpu().numpy())
            all_targets.extend(binary_gt.detach().cpu().numpy())

            e_binary  += binary_loss.item()
            e_proto_f += proto_loss_f.item()
            e_proto_b += proto_loss_b.item()

            pbar.set_postfix(
                bin=f"{binary_loss.item():.3f}",
                pf=f"{proto_loss_f.item():.3f}",
                pb=f"{proto_loss_b.item():.3f}",
                pwf=f"{pw_f:.2f}",
                pwb=f"{pw_b:.2f}",
            )

        n              = max(len(train_loader), 1)
        train_acc      = accuracy_score(all_targets, all_preds)
        train_macro_f1 = f1_score(all_targets, all_preds, average="macro",    zero_division=0)
        train_wt_f1    = f1_score(all_targets, all_preds, average="weighted", zero_division=0)

        print(f"\n{'='*70}")
        print(
            f"Epoch {epoch+1}/{epochs}  "
            f"binary={e_binary/n:.4f}  "
            f"proto_f={e_proto_f/n:.4f}  proto_b={e_proto_b/n:.4f}  "
            f"pwf={pw_f:.3f}  pwb={pw_b:.3f}  "
            f"train_acc={train_acc:.4f}  train_macro_f1={train_macro_f1:.4f}"
        )
        print(
            f"Proto finding updates: {model.proto_finding_updates.cpu().tolist()}  "
            f"| Proto birads  updates: {model.proto_birads_updates.cpu().tolist()}"
        )
        print("\n--- Train (Binary) ---")
        print(classification_report(all_targets, all_preds, target_names=class_names, zero_division=0))

        val_metrics = validate(
            model=model,
            loader=val_loader,
            device=device,
            binary_loss_fn=binary_loss_fn,
            class_names=class_names,
        )

        print(
            f"Val loss={val_metrics['loss']:.4f}  "
            f"Val acc={val_metrics['acc']:.4f}  "
            f"Val macro_f1={val_metrics['macro_f1']:.4f}  "
            f"Val weighted_f1={val_metrics['weighted_f1']:.4f}"
        )
        print('=' * 70)

        scheduler.step()

        with open(log_path, "a") as fh:
            fh.write(
                f"Epoch {epoch+1}  "
                f"binary={e_binary/n:.4f}  "
                f"proto_f={e_proto_f/n:.4f}  proto_b={e_proto_b/n:.4f}  "
                f"pwf={pw_f:.3f}  pwb={pw_b:.3f}  "
                f"train_acc={train_acc:.4f}  train_macro_f1={train_macro_f1:.4f}  "
                f"train_wt_f1={train_wt_f1:.4f}  "
                f"val_loss={val_metrics['loss']:.4f}  "
                f"val_acc={val_metrics['acc']:.4f}  "
                f"val_macro_f1={val_metrics['macro_f1']:.4f}  "
                f"val_wt_f1={val_metrics['weighted_f1']:.4f}\n"
            )

        if val_metrics["macro_f1"] > best_macro_f1:
            best_macro_f1 = val_metrics["macro_f1"]
            torch.save({
                "epoch":            epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state":  optimizer.state_dict(),
                "best_macro_f1":    best_macro_f1,
            }, save_path)
            print(f"Saved best model macro-F1={best_macro_f1:.4f}")
            not_improved = 0
        else:
            not_improved += 1
            if not_improved >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break

    return best_macro_f1


# =============================================================================
# Runner
# =============================================================================

def run_experiments(
    train_loader,
    val_loader,
    test_loader,
    inbreast_loader,
):
    seed_everything(42)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    train_config = dict(
        lr_backbone   = 5e-5,
        lr_heads      = 5e-5,
        epochs        = 60,
        patience      = 15,
        w_finding     = 1.0,
        w_birads      = 1.0,
        warmup_epochs = 3,
        ramp_epochs   = 10,
        class_names   = BINARY_CLASS_NAMES,
    )

    backbones = [

        {
            "name": "efficientnet_b3",
            "fn":   models.efficientnet_b3,
            "w":    models.EfficientNet_B3_Weights.DEFAULT,
        },
        {
            "name": "convnext_base",
            "fn":   models.convnext_base,
            "w":    models.ConvNeXt_Base_Weights.DEFAULT,
        },

    ]

    all_results = {}

    for cfg in backbones:
        out_dir   = f"/workspace/Thesis_updated_results/binary_512_hard_dual_proto/{cfg['name']}"
        save_path = f"{out_dir}/best_model.pth"
        os.makedirs(out_dir, exist_ok=True)

        print(f"\n{'#'*70}")
        print(f"Backbone: {cfg['name']}")
        print(f"{'#'*70}")
        print(f"Binary class weights:     {BINARY_CLASS_WEIGHT}")
        print(f"Finding proto momentums:  {FINDING_PROTO_MOMENTUM}")
        print(f"BI-RADS  proto momentums: {BIRADS_PROTO_MOMENTUM}")
        print(f"\nEnhancements enabled:")
        print(f"  - Hard Negative Mining: {HARD_NEG_MINING_ENABLED} (start epoch {HARD_NEG_MINING_START_EPOCH}, threshold {HARD_NEG_THRESHOLD})")
        print(f"  - Mixup Augmentation:   {MIXUP_ENABLED} (alpha {MIXUP_ALPHA})")

        model = DualProtoNet(
            backbone_name    = cfg["name"],
            backbone_fn      = cfg["fn"],
            backbone_weights = cfg["w"],
            emb_dim          = 128,
            dropout          = 0.4,
            num_classes      = NUM_CLASSES,
            num_findings     = NUM_FINDINGS,
            temperature      = 0.1,
        ).to(device)

        num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"Trainable params: {num_params:,}")

        best_macro_f1 = train_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            device=device,
            save_path=save_path,
            **train_config,
        )

        ckpt = torch.load(save_path, map_location=device)
        model.load_state_dict(ckpt["model_state_dict"])
        print(f"Loaded epoch {ckpt['epoch']+1} | best macro-F1={ckpt['best_macro_f1']:.4f}")

        vindr_metrics = test_model(
            model=model,
            loader=test_loader,
            device=device,
            save_dir=out_dir,
            name="VinDr-Test",
            class_names=BINARY_CLASS_NAMES,
        )

        inbreast_metrics = test_model(
            model=model,
            loader=inbreast_loader,
            device=device,
            save_dir=out_dir,
            name="INbreast",
            class_names=BINARY_CLASS_NAMES,
        )

        all_results[cfg["name"]] = dict(
            best_val_macro_f1    = best_macro_f1,
            vindr_acc            = vindr_metrics["acc"],
            vindr_macro_f1       = vindr_metrics["macro_f1"],
            vindr_weighted_f1    = vindr_metrics["weighted_f1"],
            inbreast_acc         = inbreast_metrics["acc"],
            inbreast_macro_f1    = inbreast_metrics["macro_f1"],
            inbreast_weighted_f1 = inbreast_metrics["weighted_f1"],
        )

        del model, ckpt
        torch.cuda.empty_cache()
        gc.collect()

    print(f"\n{'='*85}")
    print("FINAL RESULTS")
    print(f"{'='*85}")
    print(
        f"{'Model':<22} "
        f"{'Val Macro-F1':>12} "
        f"{'VinDr Macro-F1':>15} "
        f"{'INbreast Macro-F1':>18}"
    )
    print("-" * 85)
    for name, r in all_results.items():
        print(
            f"{name:<22} "
            f"{r['best_val_macro_f1']:>12.4f} "
            f"{r['vindr_macro_f1']:>15.4f} "
            f"{r['inbreast_macro_f1']:>18.4f}"
        )

    return all_results

results = run_experiments(
    train_loader    = tr_dl,
    val_loader      = val_dl,
    test_loader     = ts_dl,
    inbreast_loader = inbreast_dl,
)

Device: cuda

######################################################################
Backbone: efficientnet_b3
######################################################################
Binary class weights:     [0.7313432835820896, 1.5806451612903225]
Finding proto momentums:  [0.999, 0.997, 0.98, 0.96, 0.92, 0.85]
BI-RADS  proto momentums: [0.999, 0.997, 0.97, 0.96, 0.92]

Enhancements enabled:
  - Hard Negative Mining: True (start epoch 5, threshold 0.45)
  - Mixup Augmentation:   True (alpha 1.0)
Trainable params: 16,802,987


Epoch 1/60: 100%|██████████| 441/441 [06:37<00:00,  1.11it/s, bin=0.070, pb=0.000, pf=0.000, pwb=0.00, pwf=0.00] 


Epoch 1/60  binary=0.1033  proto_f=0.0000  proto_b=0.0000  pwf=0.000  pwb=0.000  train_acc=0.4257  train_macro_f1=0.4167
Proto finding updates: [441, 433, 284, 127, 66, 17]  | Proto birads  updates: [441, 434, 177, 185, 80]

--- Train (Binary) ---
                        precision    recall  f1-score   support

  Negative (BI-RADS 1)       0.78      0.22      0.34      4823
Positive (BI-RADS 2-5)       0.34      0.87      0.49      2233

              accuracy                           0.43      7056
             macro avg       0.56      0.54      0.42      7056
          weighted avg       0.64      0.43      0.39      7056




--- Validation (Binary) ---
                        precision    recall  f1-score   support

  Negative (BI-RADS 1)       0.76      0.72      0.74       538
Positive (BI-RADS 2-5)       0.46      0.52      0.49       249

              accuracy                           0.66       787
             macro avg       0.61      0.62      0.62       787
          weighted avg       0.67      0.66      0.66       787

Val loss=0.1079  Val acc=0.6582  Val macro_f1=0.6163  Val weighted_f1=0.6629
Saved best model macro-F1=0.6163


Epoch 2/60: 100%|██████████| 441/441 [05:26<00:00,  1.35it/s, bin=0.049, pb=0.000, pf=0.000, pwb=0.00, pwf=0.00]


...

Epoch 26/60: 100%|██████████| 441/441 [06:25<00:00,  1.14it/s, bin=0.044, pb=1.646, pf=2.636, pwb=1.00, pwf=1.00]



Epoch 26/60  binary=0.0493  proto_f=1.0877  proto_b=0.8615  pwf=1.000  pwb=1.000  train_acc=0.8611  train_macro_f1=0.8451
Proto finding updates: [11466, 11311, 7128, 3288, 1732, 462]  | Proto birads  updates: [11466, 11317, 4726, 5014, 1975]

--- Train (Binary) ---
                        precision    recall  f1-score   support

  Negative (BI-RADS 1)       0.93      0.87      0.89      4823
Positive (BI-RADS 2-5)       0.75      0.85      0.80      2233

              accuracy                           0.86      7056
             macro avg       0.84      0.86      0.85      7056
          weighted avg       0.87      0.86      0.86      7056


--- Validation (Binary) ---
                        precision    recall  f1-score   support

  Negative (BI-RADS 1)       0.85      0.85      0.85       538
Positive (BI-RADS 2-5)       0.67      0.68      0.68       249

              accuracy                           0.80       787
             macro avg       0.76      0.77      0.76      

Epoch 27/60: 100%|██████████| 441/441 [06:54<00:00,  1.06it/s, bin=0.070, pb=1.984, pf=0.461, pwb=1.00, pwf=1.00]



Epoch 27/60  binary=0.0475  proto_f=1.0788  proto_b=0.8170  pwf=1.000  pwb=1.000  train_acc=0.8635  train_macro_f1=0.8477
Proto finding updates: [11907, 11748, 7395, 3412, 1798, 480]  | Proto birads  updates: [11907, 11754, 4904, 5206, 2052]

--- Train (Binary) ---
                        precision    recall  f1-score   support

  Negative (BI-RADS 1)       0.93      0.87      0.90      4823
Positive (BI-RADS 2-5)       0.75      0.85      0.80      2233

              accuracy                           0.86      7056
             macro avg       0.84      0.86      0.85      7056
          weighted avg       0.87      0.86      0.87      7056


--- Validation (Binary) ---
                        precision    recall  f1-score   support

  Negative (BI-RADS 1)       0.85      0.89      0.87       538
Positive (BI-RADS 2-5)       0.73      0.65      0.69       249

              accuracy                           0.81       787
             macro avg       0.79      0.77      0.78      

Epoch 28/60:  71%|███████   | 313/441 [08:35<02:22,  1.12s/it, bin=0.060, pb=1.205, pf=1.985, pwb=1.00, pwf=1.00]

In [ ]:
import torch
import torchvision.models as models

def inspect_convnext():
    model = models.convnext_base(weights=models.ConvNeXt_Base_Weights.DEFAULT)
    model.eval()

    print("\n=== ConvNeXt Architecture ===\n")
    print(model)

    x = torch.randn(1, 3, 512, 512)

    feats = model.features

    print("\n=== Forward Feature Shapes ===\n")

    x = feats[0](x)
    print("Stem:", x.shape)

    x = feats[1](x)
    print("Stage1:", x.shape)

    x = feats[2](x)
    print("Down1:", x.shape)

    x = feats[3](x)
    print("Stage2:", x.shape)

    x = feats[4](x)
    print("Down2:", x.shape)

    p4 = feats[5](x)
    print("Stage3 (P4 - MID):", p4.shape)

    x = feats[6](p4)
    print("Down3:", x.shape)

    p5 = feats[7](x)
    print("Stage4 (P5 - HIGH):", p5.shape)

inspect_convnext()

In [ ]:
def inspect_efficientnet():
    model = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.DEFAULT)
    model.eval()

    print("\n=== EfficientNet-B3 Architecture ===\n")
    print(model)

    x = torch.randn(1, 3, 512, 512)

    feats = model.features

    print("\n=== Forward Feature Shapes ===\n")

    for i, layer in enumerate(feats):
        x = layer(x)
        print(f"Layer {i}: {x.shape}")

inspect_efficientnet()

In [ ]:
jjjjjjjjjjjj

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models

# -----------------------------
# Dummy Attention Pool (if not defined)
# -----------------------------
class AttentionPool2d(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Conv2d(in_dim, in_dim // 4, 1),
            nn.ReLU(),
            nn.Conv2d(in_dim // 4, 1, 1)
        )

    def forward(self, x):
        # x: [B, C, H, W]
        attn = self.attn(x)                 # [B, 1, H, W]
        attn = attn.flatten(2)              # [B, 1, HW]
        attn = torch.softmax(attn, dim=-1)  # attention weights

        x = x.flatten(2)                    # [B, C, HW]
        out = (x * attn).sum(-1)            # weighted sum → [B, C]
        return out


# -----------------------------
# Your Model (unchanged)
# -----------------------------
class FindingAwareModel(nn.Module):
    def __init__(self, backbone_name, backbone, emb_dim=128,
                 dropout=0.4, num_classes=2):
        super().__init__()
        self.backbone_name = backbone_name.lower()
        self.backbone      = backbone

        if "efficientnet" in self.backbone_name:
            self.num_features   = backbone.classifier[1].in_features
            backbone.classifier[1] = nn.Identity()
            self.is_cnn         = True
        elif "convnext" in self.backbone_name:
            self.num_features   = backbone.classifier[2].in_features
            backbone.classifier[2] = nn.Identity()
            self.is_cnn         = True
        elif "swin" in self.backbone_name:
            self.num_features = backbone.head.in_features
            backbone.head     = nn.Identity()
            self.is_cnn       = False
        else:
            raise ValueError(backbone_name)

        self.pool = AttentionPool2d(self.num_features) if self.is_cnn else None

        self.classifier_head = nn.Sequential(
            nn.Linear(self.num_features, 512),
            nn.LayerNorm(512), nn.GELU(),
            nn.Dropout(dropout / 2),
            nn.Linear(512, num_classes),
        )

        self.proj_head = nn.Sequential(
            nn.Linear(self.num_features, self.num_features),
            nn.BatchNorm1d(self.num_features), nn.GELU(),
            nn.Linear(self.num_features, emb_dim),
        )

    def _extract(self, x):
        f = self.backbone(x)

        print(f"[Backbone Output Shape]: {f.shape if not isinstance(f, tuple) else 'tuple'}")

        if isinstance(f, tuple):
            f = f[0]

        if self.is_cnn:
            if f.ndim == 4:
                print("[CNN Feature Map]:", f.shape)
                f = self.pool(f) if self.pool else f.flatten(2).mean(-1)
            elif f.ndim == 3:
                f = f.mean(1)
        else:
            if f.ndim == 3:
                f = f.mean(1)
            elif f.ndim == 4:
                f = f.flatten(2).mean(-1)

        print("[Final Feature Vector f]:", f.shape)
        return f

    def forward(self, x, return_embedding=False):
        f = self._extract(x)

        logits = self.classifier_head(f)
        print("[Logits Shape]:", logits.shape)

        if return_embedding:
            emb = F.normalize(self.proj_head(f), dim=1)
            print("[Embedding Shape]:", emb.shape)
            return logits, emb

        return logits


# -----------------------------
# TESTING FUNCTION
# -----------------------------
def test_model(backbone_name):
    device = "cuda" if torch.cuda.is_available() else "cpu"

    if backbone_name == "efficientnet":
        backbone = models.efficientnet_b3(weights=None)
    elif backbone_name == "convnext":
        backbone = models.convnext_base(weights=None)
    else:
        raise ValueError

    model = FindingAwareModel(backbone_name, backbone).to(device)

    # Try BOTH resolutions
    for img_size in [512]:
        print("\n" + "="*50)
        print(f"Testing {backbone_name.upper()} with input {img_size}x{img_size}")
        print("="*50)

        x = torch.randn(2, 3, img_size, img_size).to(device)

        with torch.no_grad():
            logits, emb = model(x, return_embedding=True)


# -----------------------------
# RUN TESTS
# -----------------------------
test_model("efficientnet")
test_model("convnext")

In [ ]:
ssssssss

In [ ]:

from tqdm import tqdm
import torch.nn as nn
from torchvision import models
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
from torch.optim import Adam
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
fffffff

In [ ]:
print("h1")